In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import ADASYN
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('tourism_data.csv ')
df

In [ ]:
df.info()

In [ ]:
# Drop unnecessary columns
drop_cols = ['TransactionId', 'AttractionId', 'AttractionAddress', 'ContenentId', 'CountryId', 'RegionId','AttractionTypeId','CityId','AttractionCityId','UserId']
df.drop(columns=drop_cols, inplace=True)

In [ ]:
df.info()

In [ ]:
# Label Encoding for Categorical Columns
label_encoders = {}
categorical_cols = ['Attraction', 'AttractionType', 'CityName', 'Contenent', 'Country', 'Region', 'VisitMode']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
# Handling outliers using IQR method
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

In [ ]:
df.info()


In [ ]:
# Splitting data into features and target
X = df.drop(columns=['VisitMode'])
y = df['VisitMode']

In [ ]:
# Scaling numerical features
scaler = MinMaxScaler()
X_SCALED = scaler.fit_transform(X)

In [ ]:
from imblearn.over_sampling import ADASYN
adasyn = ADASYN(random_state=42)
X_resampled, y_resampled = adasyn.fit_resample(X_SCALED, y)

In [ ]:
# Proceed with train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42)

In [ ]:
# Initialize XGBoost Classifier
model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')


In [ ]:
# Hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters
best_params = grid_search.best_params_
print("Best Parameters:", best_params)

In [ ]:
# Train final model with best parameters
best_model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='mlogloss')
best_model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = best_model.predict(X_test)


In [ ]:
# Model Evaluation
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import joblib

joblib.dump(best_model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')